# Automated Skill Discovery with RLMs

This notebook accompanies Chapter 10 §10.3.1 of *Context Engineering with DSPy*: Automated Skill Discovery with RLMs.

## Required environment variables

Add these to a `.env` at the repo root (see `.env.example`):

- `OPENAI_API_KEY` — used as the task / judge / router LM via `dspy.LM("openai/...")`.
- `OPENROUTER_API_KEY` — used when the notebook calls Anthropic models via OpenRouter (`dspy.LM("openrouter/anthropic/claude-opus-4.7")`). No Claude Code CLI is required.

## Original source

Imported from the writing repo at `working/skill-discovery-rlm.ipynb`.


In [ ]:
%pip install -r ../requirements.txt -q

# Discovering new skills from past sessions with an RLM

When users repeatedly ask an agent for the same kind of help, that's a missing skill. The catch is you don't notice because each session looks different on the surface. A Recursive Language Model can read a directory of past sessions, cluster them by intent, and propose a `SKILL.md` for the most common cluster.

**How it works:**
- **Sessions** — a small directory of past agent transcripts.
- **RLM** — given the directory, writes Python to read each transcript, summarize it via a sub-LLM, cluster by intent, and pick the largest cluster.
- **Output** — a candidate `SKILL.md` written from the patterns the RLM found.

This is the same loop a human would run -- read sessions, spot the pattern, write the skill -- but the RLM does the reading and clustering for you so you only have to review the proposal.

**Prerequisites:**
- `OPENAI_API_KEY` in your environment
- Deno installed for the default Python interpreter (`brew install deno`)

In [ ]:
import os, json, dspy
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY first"

dspy.configure(lm=dspy.LM("openai/gpt-5-mini", temperature=0.7, max_tokens=8000))

## Mock a session directory

In production this would be the agent's actual session log directory (`~/.claude/projects/<repo>/sessions/`, or wherever your harness writes transcripts). For the notebook we'll write five short fake sessions to a temp directory. Four of them are the same underlying request -- "port this Postgres SQL to BigQuery" -- worded differently each time. One is unrelated.

In [ ]:
SESSIONS = {
    "2026-04-12-billing.md": """User: this query runs in postgres but bigquery rejects it, can you fix
```sql
SELECT user_id, EXTRACT(EPOCH FROM created_at) FROM events
WHERE created_at > NOW() - INTERVAL '30 days' LIMIT 100;
```
Assistant: replaced EXTRACT(EPOCH...) with UNIX_SECONDS(created_at) and NOW()-INTERVAL with TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY).
""",
    "2026-04-14-funnel.md": """User: bigquery is complaining about this, came from our analytics db
```sql
SELECT date_trunc('day', ts), count(*) FROM signups GROUP BY 1;
```
Assistant: bigquery uses TIMESTAMP_TRUNC, not date_trunc. swapped it and aliased the column.
""",
    "2026-04-15-cohorts.md": """User: porting these postgres queries to BQ for the dashboard, this one breaks
```sql
SELECT user_id, ARRAY_AGG(event ORDER BY ts) FROM events GROUP BY user_id;
```
Assistant: BQ supports ARRAY_AGG but ORDER BY goes inside as ARRAY_AGG(event ORDER BY ts ASC).
""",
    "2026-04-16-retention.md": """User: another postgres-isms one, lateral join doesn't work in bigquery
```sql
SELECT u.id, e.first_seen FROM users u, LATERAL (SELECT MIN(ts) AS first_seen FROM events WHERE user_id = u.id) e;
```
Assistant: rewrote as a correlated subquery: SELECT u.id, (SELECT MIN(ts) FROM events WHERE user_id = u.id) AS first_seen FROM users u.
""",
    "2026-04-17-css.md": """User: hero section is wider than the viewport on mobile, no clue why
Assistant: the .grid container has min-width: 1200px. removed it and added overflow-x: hidden on body as a safety net.
""",
}

sessions_dir = Path("./fake_sessions"); sessions_dir.mkdir(exist_ok=True)
for name, body in SESSIONS.items():
    (sessions_dir / name).write_text(body)

print(f"wrote {len(SESSIONS)} sessions to {sessions_dir}/")

## Let the RLM read and cluster

`dspy.RLM` gives the model a Python REPL plus `llm_query()` for semantic work. We hand it the list of session paths and a signature that says: find the dominant intent, then write a skill for it. The RLM reads each file in code, asks a sub-LLM to summarize each one's intent, clusters the summaries, and only the largest cluster's skill comes back.

In [ ]:
class DiscoverSkill(dspy.Signature):
    """Read every session file in `session_paths`. For each file, summarize
    in one sentence what the user was asking the agent to do. Group similar
    summaries. For the largest group (the dominant recurring intent), write
    a SKILL.md that would let the agent handle that request directly next
    time -- with a YAML frontmatter (name, description) and a body of concrete
    rules drawn from how the assistant actually responded in those sessions.

    Skip groups smaller than 2 sessions. Return only the SKILL.md text.
    """
    session_paths: list[str] = dspy.InputField()
    skill_md: str = dspy.OutputField()

rlm = dspy.RLM(DiscoverSkill, max_iterations=8, max_llm_calls=10)
result = rlm(session_paths=[str(p) for p in sorted(sessions_dir.glob("*.md"))])

print(result.skill_md)

## Review and install

The RLM proposes; you decide. Read the candidate, edit anything that's off, then save it to wherever your harness picks up skills (`~/.claude/skills/<name>/SKILL.md`, `.cursor/rules/`, etc.). From here the Section 10.2 loop takes over -- run the new skill on real briefs, capture the failures, and feed those failures into `optimize_anything` to evolve it.

In [ ]:
Path("./proposed_skill.md").write_text(result.skill_md)
print("saved to ./proposed_skill.md -- review before installing.")